In [ ]:
#Creación Base de Datos

try:
    cursor = connection.cursor()
    query_crear_bbdd = """
    CREATE DATABASE IF NOT EXISTS musicstream
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_unicode_ci
    """
    cursor.execute(query_crear_bbdd)
    print("Query existosa")
except Error as e:
    print(e)

In [ ]:
cursor.execute("use musicstream")
query_crear_tabla = '''CREATE TABLE artistas (
                        id_artista INT PRIMARY KEY AUTO_INCREMENT, 
                        nombre_artista VARCHAR(50) NOT NULL
                        );'''
cursor.execute(query_crear_tabla)
print ("Tabla creada correctamente")

In [ ]:
df_tabla_artistas = pd.read_csv(
    "artistas_completo_deezer.csv",
    encoding="utf-8"
)
df_tabla_artistas = df_tabla_artistas[["nombre_artista"]].drop_duplicates()
for index, fila in df_tabla_artistas.iterrows():
    query_insert = """
    INSERT INTO artistas (nombre_artista)
    VALUES (%s)
    """
    valores = (fila["nombre_artista"],)
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")

In [ ]:
cursor.execute("use musicstream")
query_crear_tabla = """
CREATE TABLE canciones (
    id_cancion INT PRIMARY KEY AUTO_INCREMENT,
    id_artista INT NOT NULL,
    titulo_cancion VARCHAR(300) NOT NULL,
    titulo_album VARCHAR(300),
    tipo VARCHAR(100),
    año_lanzamiento INT,
    genero VARCHAR(100),
    id_genero INT,
    FOREIGN KEY (id_artista)
    REFERENCES artistas(id_artista)
);
"""
cursor.execute(query_crear_tabla)
print("Tabla canciones creada correctamente")

In [ ]:
df_tabla_canciones = pd.read_csv(
    "artistas_completo_deezer.csv",
    encoding="utf-8"
)
for index, fila in df_tabla_canciones.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    id_artista = resultado[0]
    query_insert = """
    INSERT INTO canciones (
        id_artista,
        titulo_cancion,
        titulo_album,
        tipo,
        año_lanzamiento,
        genero,
        id_genero
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["titulo_cancion"],
        fila["titulo_album"],
        fila["tipo"],
        fila["año_lanzamiento"],
        fila["genero"]
        if pd.notna(fila["genero"])
        else None,
        fila["id_genero"]
        if pd.notna(fila["id_genero"])
        else None
    )
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")

In [ ]:
query_crear_tabla = """
CREATE TABLE informacion_artistas (
    id_info INT PRIMARY KEY AUTO_INCREMENT,
    id_artista INT NOT NULL,
    nombre_artista VARCHAR(50),
    biografia TEXT,
    listeners BIGINT,
    playcount BIGINT,
    artistas_similares TEXT,
    FOREIGN KEY (id_artista)
    REFERENCES artistas(id_artista)
);
"""
cursor.execute(query_crear_tabla)
print("Tabla informacion_artistas creada correctamente")
 

In [ ]:
df_lfm = pd.read_csv(
    "artistas_completo_LFM.csv",
    encoding="utf-8"
)
for index, fila in df_lfm.iterrows():
    query_id_artista = """
    SELECT id_artista
    FROM artistas
    WHERE nombre_artista = %s
    """
    cursor.execute(
        query_id_artista,
        (fila["nombre_artista"],)
    )
    resultado = cursor.fetchone()
    id_artista = resultado[0]
    query_insert = """
    INSERT INTO informacion_artistas (
        id_artista,
        nombre_artista,
        biografia,
        listeners,
        playcount,
        artistas_similares
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    valores = (
        id_artista,
        fila["nombre_artista"],
        fila["biografia"],
        fila["listeners"],
        fila["playcount"],
        fila["artistas_similares"]
    )
    cursor.execute(query_insert, valores)
connection.commit()
print("Datos insertados correctamente")